# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [2]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [4]:
dataset["train"][118]

{'label': 0,
 'text': "This is the absolute WORST Steak N Shake I've ever been to. \\n\\nThe bf and I got lost around Pittsburgh for 40 min. trying to find this location and on top of the unnecessary 1 hour wait for our food, the food itself was just so crappily made. It was so bad and we had waited a ridiculous amount of time (c'mon, Steak N Shake is basically a glorified McD's), our waiter comped our WHOLE MEAL.\\n\\nIf not for the compensation, I would have been pissed that we had to pay for such a crappy meal. The bf, on the other hand, was more upset about the principle of the matter... we had trekked all that way just to get our Steak N Shake fix (since both of us haven't had it since our undergraduate days) and it was a major fail.\\n\\nI think I'm good with my Steak N Shake craving for a long time now...\\n\\n\\n\\nPS. The whole joint smells like major B.O."}

In [5]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [6]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [7]:
show_random_elements(dataset["train"])

,label,text
0,2 star,"Cool little place, but we had mixed reviews on the food. Picked up an order to-go recently and had an order of the pulled pork and a slab of ribs. The pork was pretty darn good and came with giant buns to make a delicious pork sandwich. We all agreed though that the BBQ sauce that came on the side was NOT good at all. Not sure what was wrong, but the taste was way off. We ended up using some bottled sauce from the fridge instead. The ribs, while meaty and pretty tender, were nothing special and again better without the sauce. This dish came with a side of the spicy BBQ sauce which was just as odd as the regular - just spicier! All in all, we might give it a shot again sometime down the road, but for now I think we'll look elsewhere for a BBQ-fix."
1,5 stars,Sonoran Dogs-Yum! Horchata.... Bottled Mexican sodas....this place makes me miss trips to mexico
2,1 star,This place is the worst strip club in the world I would avoid this place at all costs I would avoid this place. I think that the person in the front is the most rude ass person in the world. All we asked was to speak with the manager an got escorted out
3,5 stars,"What a great find!! I'm not from the area but I've been looking for a place like this for a while. I think there should be a Zzeeks on every corner.\n\nSuper friendly, cheap, and GOOD! Awesome atmosphere fully loaded with 5 big screens for the games! Perfect size establishment and an experience you don't want to leave. Got the $5 lunch special complete with a large slice (1/4 of a 17\"" pizza) and a fountain drink. Ate there but was given a Togo cup when I was ready to leave. Make sure to ask for a side of ranch to dip, it's on the house! Will come back, even if I have to drive from Mesa to do so.\nThanks for an awesome Sunday lunch Zzeeks!!"
4,3 stars,"Solid Dive bar. No food, very dark, and a sign in the front entrance that tells bikers not to wear their \""colors\"" -- gives it that extra dangerous feel. Bands on the weekends. Sarcastic bartenders will trade barbs with you all day or night. Grab some tacos next door (try the cabeza) and go back for more drinks. Enjoyed many a last stop here with my boy Dahmer, and was never disappointed."
5,4 stars,"This place is an excellent place to go if you are in or around the Strip. The food is delicious, prices are good, and staff is friendly. Although i have only had the chicken, you cannot go wrong with it. I have had the 1/2 chicken and also the crispy chicken burrito and both were very good. In addition, I always get the yucca fries; my lunch order is now a burrito with yucca fries and all for under $10 and i can't finish it all. In addition, always get the green sauce and use it to your heat tolerance. It definitely has a kick to it, so mix it with some mayo especially for the yucca fries. I don't know if I would drive out of my way to eat here, but it is worth going if you are close."
6,1 star,"MUST READ: I walked in with a question about a cut and color correction and Misty quoted me a price for correction and then for added color. She said she would throw in the cut and could get me in and out in about an hour (This was a spur of the moment stop and I didn't have a lot of time, and no one else was in shop.) As the first hour turned into the second hour, I was growing a little agitated. My stylist Misty left me at one point to correct a cut for the other stylist and then left me again to cut a wig on another client. I pointed out that she missed a spot on my roots and she said she didn't see anything. She washed my color out in the same bowl she had just done a quick bleach correction treatment in. I also asked her to get the dye off my ears as I could feel when she applied color she was slathering it all over my ears and neck along with my hair and she started to become rude with me, saying there was nothing there (fyi- I came home with a big dye mark on the right side of my neck right under my ear and also on the left side of my forehea

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/tiger/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,1 star,"As many of the reviews express there are some nice furnishings at Mod. The salespeople can be friendly when you actually talk to them in person but this is where the experience ends. \nI've been shopping the entire Valley for a sofa and found one there I actually liked at a fair price. A new one would have taken up to two months so I asked about the floor model and offered to let them keep displaying for three weeks till i moved to new place. \""we can't sell floor models\"" the sales manager said although all over the store there are Sold signs on floor models. \nHe gave me his card as I left and said to email any questions. \nThe next day I did. After a week there was no response so I called and talked to another designer/salesperson. I asked for info on manufacturer, dimensions and style. I was promised an email response. A day later nothing so I called and finally got the info sent. \nI wanted to sit on it again so I returned to the store and talked to the designer that sent the info after multiple requests. I was told another one had shown up in their warehouse in a different color with awful accent pillows. I said I would think about it. \nNext day I asked them to give me a price I can't refuse offer given all that had transpired and if good enough I would buy. Four days later no response so I asked if email was received. They said yes but wouldn't negotiate. I then gave a reasonable offer within 15% of asking price and still have not heard a word for many days. \nThe company will not exist long if they don't do all they can to both follow up with customers and realize they won't have a job if they don't sell everything they can. The management needs to wake up and start doing some sales training as each time I was there the salespeople were more interested in ordering or eating lunch than making a sale. They also need to know that with steep furniture competition (two stores right next to them) they won't last long as the only company that doesn't negotiate. \nThere is a win win in every situation and business's that don't car about customer satisfaction don't win.","[101, 1249, 1242, 1104, 1103, 3761, 6848, 1175, 1132, 1199, 3505, 26690, 1120, 12556, 1181, 119, 1109, 3813, 21159, 1169, 1129, 4931, 1165, 1128, 2140, 2037, 1106, 1172, 1107, 1825, 1133, 1142, 1110, 1187, 1103, 2541, 3769, 119, 165, 183, 2240, 112, 1396, 1151, 6001, 1103, 2072, 2634, 1111, 170, 10907, 1105, 1276, 1141, 1175, 146, 2140, 3851, 1120, 170, 4652, 3945, 119, 138, 1207, 1141, 1156, 1138, 1678, 1146, 1106, 1160, 1808, 1177, 146, 1455, 1164, 1103, 1837, 2235, 1105, 2356, 1106, 1519, 1172, 1712, 15959, 1111, 1210, 2277, 6174, 178, 1427, 1106, 1207, 1282, 119, 165, 107, 1195, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [10]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))
len(small_eval_dataset)

1000

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [11]:
from tqdm import tqdm
import time

# 降级后，这个交互式版本现在应该可以完美工作了
for i in tqdm(range(10), desc="测试进度条"):
    time.sleep(0.5)

测试进度条: 100%|██████████| 10/10 [00:05<00:00,  2.00it/s]


In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [13]:
from transformers import TrainingArguments

model_dir = "../models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  per_device_train_batch_size=32,
                                  num_train_epochs=5,
                                  logging_steps=10000)

In [14]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_better=None,
group_by_le

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [15]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

In [17]:
from transformers import TrainingArguments, Trainer
import os
training_args = TrainingArguments(
                                    output_dir=model_dir,
                                    evaluation_strategy="epoch", 
                                    save_strategy="epoch",
                                    load_best_model_at_end=True,
                                    metric_for_best_model="accuracy",     # 若以准确率择优
                                    greater_is_better=True,               # 准确率越大越好
                                    save_total_limit=1,                   # 仅保留1个checkpoint（最优）
                                    save_only_model=True,                 # 只保存模型权重，减小体积

                                    per_device_train_batch_size=32,
                                    num_train_epochs=3,
                                    report_to=["tensorboard"],
                                    logging_dir=os.path.join(model_dir, "tb"),
                                    logging_steps=10000,
                                    disable_tqdm=True)

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)
smaill_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 1 nvidia-smi`:

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 14:37:41 2023

Wed Dec 20 14:37:41 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   64C    P0              69W /  70W |   6665MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     18395      C   /root/miniconda3/bin/python                6660MiB |
+---------------------------------------------------------------------------------------+
```
```shell
Wed Mar 18 00:02:42 2026
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A10                     On  | 00000000:65:01.0 Off |                  Off |
|  0%   62C    P0             155W / 150W |  22010MiB / 24564MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A   2093349      C   ...er/miniforge3/envs/py310/bin/python    22002MiB |
+---------------------------------------------------------------------------------------+
```


In [59]:
smaill_trainer.train()

{'eval_loss': 1.3623497486114502, 'eval_accuracy': 0.364, 'eval_runtime': 12.5238, 'eval_samples_per_second': 79.848, 'eval_steps_per_second': 9.981, 'epoch': 1.0}
{'eval_loss': 1.1834285259246826, 'eval_accuracy': 0.49, 'eval_runtime': 12.7932, 'eval_samples_per_second': 78.166, 'eval_steps_per_second': 9.771, 'epoch': 2.0}
{'eval_loss': 1.1124262809753418, 'eval_accuracy': 0.535, 'eval_runtime': 12.8031, 'eval_samples_per_second': 78.106, 'eval_steps_per_second': 9.763, 'epoch': 3.0}
{'train_runtime': 137.4878, 'train_samples_per_second': 21.82, 'train_steps_per_second': 0.698, 'train_loss': 1.3042233784993489, 'epoch': 3.0}


TrainOutput(global_step=96, training_loss=1.3042233784993489, metrics={'train_runtime': 137.4878, 'train_samples_per_second': 21.82, 'train_steps_per_second': 0.698, 'train_loss': 1.3042233784993489, 'epoch': 3.0})

In [19]:
trainer.train()

{'loss': 0.8203, 'learning_rate': 4.179507376228688e-05, 'epoch': 0.49}
{'loss': 0.7549, 'learning_rate': 3.3590147524573756e-05, 'epoch': 0.98}
{'eval_loss': 0.7253664135932922, 'eval_accuracy': 0.68266, 'eval_runtime': 638.1458, 'eval_samples_per_second': 78.352, 'eval_steps_per_second': 9.794, 'epoch': 1.0}
{'loss': 0.6802, 'learning_rate': 2.5385221286860633e-05, 'epoch': 1.48}
{'loss': 0.6689, 'learning_rate': 1.718029504914751e-05, 'epoch': 1.97}
{'eval_loss': 0.7100502848625183, 'eval_accuracy': 0.6926, 'eval_runtime': 637.5283, 'eval_samples_per_second': 78.428, 'eval_steps_per_second': 9.803, 'epoch': 2.0}
{'loss': 0.5782, 'learning_rate': 8.975368811434385e-06, 'epoch': 2.46}
{'loss': 0.5624, 'learning_rate': 7.704425737212622e-07, 'epoch': 2.95}
{'eval_loss': 0.7356773614883423, 'eval_accuracy': 0.6967, 'eval_runtime': 639.1113, 'eval_samples_per_second': 78.234, 'eval_steps_per_second': 9.779, 'epoch': 3.0}
{'train_runtime': 65767.9122, 'train_samples_per_second': 29.65, 't

TrainOutput(global_step=60939, training_loss=0.6756122908176491, metrics={'train_runtime': 65767.9122, 'train_samples_per_second': 29.65, 'train_steps_per_second': 0.927, 'train_loss': 0.6756122908176491, 'epoch': 3.0})

#全量训练最高达到 0.69 的准确率
```json
{'eval_loss': 0.7356773614883423, 'eval_accuracy': 0.6967, 'eval_runtime': 639.1113, 'eval_samples_per_second': 78.234, 'eval_steps_per_second': 9.779, 'epoch': 3.0}
```

In [20]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [21]:
#全量数据，抽样部分来验证（相对于1000，准确率从 0.54->0.65）
smaill_trainer.evaluate(small_test_dataset)

{'eval_loss': 0.7754350900650024, 'eval_accuracy': 0.67, 'eval_runtime': 1.2471, 'eval_samples_per_second': 80.184, 'eval_steps_per_second': 10.424}


{'eval_loss': 0.7754350900650024,
 'eval_accuracy': 0.67,
 'eval_runtime': 1.2471,
 'eval_samples_per_second': 80.184,
 'eval_steps_per_second': 10.424}

In [22]:
smaill_trainer.evaluate(small_test_dataset)

{'eval_loss': 0.7754350900650024, 'eval_accuracy': 0.67, 'eval_runtime': 1.2524, 'eval_samples_per_second': 79.845, 'eval_steps_per_second': 10.38}


{'eval_loss': 0.7754350900650024,
 'eval_accuracy': 0.67,
 'eval_runtime': 1.2524,
 'eval_samples_per_second': 79.845,
 'eval_steps_per_second': 10.38}

In [23]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 0.7754350900650024, 'eval_accuracy': 0.67, 'eval_runtime': 1.2448, 'eval_samples_per_second': 80.332, 'eval_steps_per_second': 10.443, 'epoch': 3.0}


{'eval_loss': 0.7754350900650024,
 'eval_accuracy': 0.67,
 'eval_runtime': 1.2448,
 'eval_samples_per_second': 80.332,
 'eval_steps_per_second': 10.443,
 'epoch': 3.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [24]:
trainer.save_model(model_dir)

In [25]:
trainer.save_state()

In [ ]:
# trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少

全量数据，抽样部分来验证（相对于1000，准确率从 0.54->0.69）